In [ ]:
import time
import cv2
import numpy as np
import os
import keyboard
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.stf6_imitation import STF6Env, StreetFighterCNN
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from pathlib import Path
from stable_baselines3.common.policies import ActorCriticPolicy
from sdlarch_rl.utils.utils import get_last_index
import pygame
import torch as th
import torch.nn as nn
from torchvision import transforms

# env = STF6Env()
train_path="./imitation/"
sub_train_path = train_path + "steps"
DETERMINISTIC=True

last_index_imitation = get_last_index(str(sub_train_path), "bc_policy", "zip")

#last_index_imitation = 35

global myEnv

def make_env():
    def _init():
        global myEnv
        myEnv = STF6Env()

        env = WarpFrame(myEnv, width=64, height=64)
        
        return env
    return _init

env = make_vec_env(make_env(), n_envs=1, vec_env_cls=DummyVecEnv)
env = VecFrameStack(env, 4)

SCALE = 10
SCREEN_WIDTH = 228 * SCALE
SCREEN_HEIGHT = 128 * SCALE

prev_keys = set()

env.reset()

could_use_ai = False

color=(0, 0, 255)

current_epoch = 0

model = None

pygame.init()

window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

# Variáveis para visualização da rede
activation_maps = {}
heatmap_overlay = None
show_activation = True
last_conv_layer = None

def get_activation(name):
    def hook(model, input, output):
        activation_maps[name] = output.detach()
    return hook

def preprocess_observation(obs):
    if obs is None:
        return None
    
    # obs tem shape (1, height, width, channels) do VecFrameStack
    if len(obs.shape) == 4:
        # Pega apenas o último frame da stack para visualização
        last_frame = obs[0, :, :, -1]  # Último canal (frame mais recente)
        
        # Converte para 3 canais (RGB)
        if last_frame.ndim == 2:
            last_frame_rgb = np.stack([last_frame] * 3, axis=-1)
        else:
            last_frame_rgb = last_frame
        
        return last_frame_rgb.astype(np.uint8)
    
    return obs

def generate_cnn_visualization(obs_tensor, policy_model, original_img_shape):
    global activation_maps
    
    if policy_model is None:
        return None
    
    try:
        obs_tensor = th.from_numpy(obs_tensor).float()
        
        if len(obs_tensor.shape) == 4 and obs_tensor.shape[-1] == 4:
            obs_tensor = obs_tensor.permute(0, 3, 1, 2)
        
        hooks = []
        activation_maps.clear()
        
        for name, module in policy_model.named_modules():
            if isinstance(module, nn.Conv2d):
                hooks.append(module.register_forward_hook(get_activation(name)))
        
        with th.no_grad():
            if hasattr(policy_model, 'features_extractor'):
                features = policy_model.features_extractor(obs_tensor)
                
                if activation_maps:
                    last_conv_key = list(activation_maps.keys())[-1]
                    last_conv_activation = activation_maps[last_conv_key]
                    
                    channel_importance = last_conv_activation.mean(dim=(2, 3))
                    
                    heatmap = th.zeros(last_conv_activation.shape[2:])
                    for i in range(last_conv_activation.shape[1]):
                        heatmap += channel_importance[0, i] * last_conv_activation[0, i]
                    
                    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
                    heatmap = heatmap.cpu().numpy()
                    
                    heatmap = cv2.resize(heatmap, (original_img_shape[1], original_img_shape[0]))
                    
                    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
                    
                    return heatmap_colored
        
    except Exception as e:
        print(f"Erro no generate_cnn_visualization: {e}")
        return None
    finally:
        for hook in hooks:
            hook.remove()
    
    return None

def overlay_heatmap(original_img, heatmap, alpha=0.5):
    if heatmap is None:
        return original_img
    
    heatmap_resized = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))
    
    heatmap_rgb = cv2.cvtColor(heatmap_resized, cv2.COLOR_BGR2RGB)
    
    combined = cv2.addWeighted(original_img, 1 - alpha, heatmap_rgb, alpha, 0)
    
    return combined

def create_simple_activation_overlay(img, action_probs):
    height, width = img.shape[:2]
    overlay = np.zeros((height, width, 3), dtype=np.uint8)
    
    colors = [
        (255, 0, 0),    # Up
        (0, 255, 0),    # Down
        (0, 0, 255),    # Left
        (255, 255, 0),  # Right
        (255, 0, 255),  # Btn1
        (0, 255, 255),  # Btn2
        (255, 255, 255) # Btn3
    ]
    
    center_x, center_y = width // 2, height // 2
    
    for i, prob in enumerate(action_probs):
        if prob > 0.1:
            angle = 2 * np.pi * i / len(action_probs)
            radius = min(width, height) // 3
            
            x = int(center_x + radius * np.cos(angle))
            y = int(center_y + radius * np.sin(angle))
            
            circle_radius = int(20 + prob * 50)
            
            cv2.circle(overlay, (x, y), circle_radius, colors[i], -1)
            
            action_labels = ['U', 'D', 'L', 'R', 'B1', 'B2', 'B3']
            cv2.putText(overlay, f"{action_labels[i]}:{prob:.2f}", 
                       (x-15, y+5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    
    alpha = 0.3
    combined = cv2.addWeighted(img, 1 - alpha, overlay, alpha, 0)
    
    return combined

def visualize_action_regions(img, action_probs):
    height, width = img.shape[:2]
    
    regions = []
    
    if action_probs[0] > 0.1:  # Up
        cv2.rectangle(img, (0, 0), (width, height//3), (0, 255, 255), 3)
        cv2.putText(img, "UP", (width//2 - 20, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
    
    if action_probs[1] > 0.1:  # Down
        cv2.rectangle(img, (0, 2*height//3), (width, height), (0, 255, 255), 3)
        cv2.putText(img, "DOWN", (width//2 - 40, height - 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
    
    if action_probs[2] > 0.1:  # Left
        cv2.rectangle(img, (0, 0), (width//3, height), (255, 255, 0), 3)
        cv2.putText(img, "LEFT", (10, height//2), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
    
    if action_probs[3] > 0.1:  # Right
        cv2.rectangle(img, (2*width//3, 0), (width, height), (255, 255, 0), 3)
        cv2.putText(img, "RIGHT", (2*width//3 + 10, height//2), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
    
    button_y = 50
    for i in range(4, 7):
        if action_probs[i] > 0.1:
            button_color = (255, 0, 255) if i == 4 else (255, 255, 0) if i == 5 else (0, 255, 255)
            button_labels = ['PUNCH', 'KICK', 'SPECIAL']
            
            cv2.circle(img, (width - 50, button_y), 20, button_color, -1)
            cv2.putText(img, button_labels[i-4], (width - 100, button_y + 5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, button_color, 2)
            button_y += 50
    
    return img

while True:
    pygame.event.pump()

    heatmap_resized = None

    action = [np.zeros(7, dtype=np.uint8)]
    action_probs = np.zeros(7, dtype=np.float32)

    text_to_display = "Count: " + str(current_epoch)

    if keyboard.is_pressed('k'):
        could_use_ai = not could_use_ai

        if could_use_ai:
            color=(0, 255, 0)
            current_epoch += 1

            if current_epoch > last_index_imitation:
                break

            latest_model_path = sub_train_path + f"/bc_policy{current_epoch}.zip"

            print("loading from: " + str(latest_model_path))
            
            model = ActorCriticPolicy.load(
                str(latest_model_path),
            )
        else:
            color=(0, 0, 255)
        
        time.sleep(0.5)

        print("k is pressed: ", could_use_ai)

    if keyboard.is_pressed('h'):
        show_activation = not show_activation
        print(f"Show activation: {show_activation}")
        time.sleep(0.3)

    if not could_use_ai:
        model = None
        # dpad
        if keyboard.is_pressed('up'):
            action[0][0] = 1
        if keyboard.is_pressed('down'):
            action[0][1] = 1
        if keyboard.is_pressed('left'):
            action[0][2] = 1
        if keyboard.is_pressed('right'):
            action[0][3] = 1
    
        # buttons
        if keyboard.is_pressed('i'):
            action[0][4] = 1
        if keyboard.is_pressed('o'):
            action[0][5] = 1
        if keyboard.is_pressed('p'): # strong kick # 11
            action[0][6] = 1
    else:
        if model is not None:
            action, _ = model.predict(obs, deterministic=DETERMINISTIC)
            action_probs = action[0].astype(np.float32)

    obs, _, _, _ = env.step(action)

    img = myEnv.render()

    if img is not None:
        
        img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))
        img_original = img.copy()
        
        obs_for_display = preprocess_observation(obs)
        
        if could_use_ai and model is not None and show_activation:
            try:
                if obs_for_display is not None:
                    img = create_simple_activation_overlay(img, action_probs)
                    
                    img = visualize_action_regions(img, action_probs)
                    
                    gray = cv2.cvtColor(img_original, cv2.COLOR_RGB2GRAY)
                    
                    edges = cv2.Canny(gray, 50, 150)
                    
                    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    
                    for contour in contours:
                        area = cv2.contourArea(contour)
                        if area > 500 and area < 2000:
                            if np.any(action_probs[:4] > 0.1):
                                contour_color = (0, 255, 255)
                            elif np.any(action_probs[4:] > 0.1):
                                contour_color = (255, 0, 255)
                            else:
                                contour_color = (0, 255, 0)
                            
                            cv2.drawContours(img, [contour], -1, contour_color, 2)
                            
                            x, y, w, h = cv2.boundingRect(contour)
                            cv2.rectangle(img, (x, y), (x + w, y + h), contour_color, 2)
                            
                            if len(action_probs) > 0:
                                max_action = np.argmax(action_probs)
                                action_labels = ['UP', 'DOWN', 'LEFT', 'RIGHT', 'PUNCH', 'KICK', 'SPECIAL']
                                if max_action < len(action_labels):
                                    cv2.putText(img, action_labels[max_action], 
                                               (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 
                                               0.7, contour_color, 2)
            
            except Exception as e:
                print(f"Error on draw: {e}")
                import traceback
                traceback.print_exc()
        
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        
        cv2.circle(img, center=(100, 100), radius=50, color=color, thickness=2)
        
        position = (50, 250)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1.5
        thickness = 2
        line_type = cv2.LINE_AA
        cv2.putText(img, text_to_display, position, font, font_scale, color, thickness, line_type)
        
        vis_status = "NETWORK VISION: ON" if show_activation else "NETWORK VISION: OFF"
        cv2.putText(img, vis_status, (50, 300), font, 1, (255, 255, 255), 2, line_type)
        
        if could_use_ai and model is not None:
            action_text = "ACTIONS: "
            action_labels = ['U', 'D', 'L', 'R', 'P', 'K', 'S']
            for i, (label, act) in enumerate(zip(action_labels, action_probs)):
                if act > 0.1:
                    action_text += f"{label}({act:.2f}) "
            
            cv2.putText(img, action_text, (50, 350), font, 1, (0, 255, 0), 2, line_type)
            
            if len(action_probs) > 0:
                max_action = np.argmax(action_probs)
                max_prob = action_probs[max_action]
                if max_prob > 0.1:
                    main_action_labels = ['UP', 'DOWN', 'LEFT', 'RIGHT', 'PUNCH', 'KICK', 'SPECIAL']
                    main_action = main_action_labels[max_action]
                    cv2.putText(img, f"MAIN: {main_action} ({max_prob:.2f})", 
                               (50, 400), font, 1, (255, 255, 0), 2, line_type)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img_pygame = np.transpose(img, (1, 0, 2))
        
        surface = pygame.surfarray.make_surface(img_pygame)

        window.blit(surface, (0, 0))
        
    pygame.display.update()

pygame.quit()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


next search...
next search...
next search...
next search...
next search...
next search...
next search...
next search...
next search...
next search...
next search...
Window founded HWND: 2953020.
reset obs shape (128, 228, 3)
loading from: ./imitation/steps/bc_policy1.zip


D:\Python311\Lib\site-packages\stable_baselines3\common\policies.py:176: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_variables = th.load(path, map_location=device)


k is pressed:  True
k is pressed:  False
loading from: ./imitation/steps/bc_policy2.zip
k is pressed:  True
k is pressed:  False
loading from: ./imitation/steps/bc_policy3.zip
k is pressed:  True
k is pressed:  False
loading from: ./imitation/steps/bc_policy4.zip
k is pressed:  True
k is pressed:  False
loading from: ./imitation/steps/bc_policy5.zip
k is pressed:  True
